In [1]:
%pip install numpy catboost pandas scikit-learn

# Import Libraries

In [2]:
import os
import re
import numpy as np
import pandas as pd
from catboost import Pool, CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

# Read Data

In [3]:
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")
print(f"Number of rows and columns in the train data set: {train_data.shape}")
print(f"Number of rows and columns in the test data set: {test_data.shape}")
train_data.head()

Number of rows and columns in the train data set: (48665, 2)
Number of rows and columns in the test data set: (12167, 2)


,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


In [4]:
train_data.groupby("rate").describe()

text                               
      count unique                top freq
rate                                      
1      4138   4130             Грязно    3
2      2410   2407  Отстойный магазин    2
3      6126   6070          Нормально    7
4      9922   9763               Норм   14
5     26069  24804    Хороший магазин  107

# Preparing the data and creating Catboost model

In [5]:
print("\nСоздание TF-IDF признаков...")
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9
)

# Создаем TF-IDF матрицы
X_train_tfidf = tfidf.fit_transform(train_data['text']).toarray()
X_test_tfidf = tfidf.transform(test_data['text']).toarray()

# Создаем DataFrame с TF-IDF признаками
tfidf_columns = [f'tfidf_{i}' for i in range(X_train_tfidf.shape[1])]
train_tfidf_df = pd.DataFrame(X_train_tfidf, columns=tfidf_columns)
test_tfidf_df = pd.DataFrame(X_test_tfidf, columns=tfidf_columns)

# Добавляем исходный текст как признак
train_data_with_features = pd.concat([train_data[['rate', 'text']], train_tfidf_df], axis=1)
test_data_with_features = pd.concat([test_data[['text']], test_tfidf_df], axis=1)



Создание TF-IDF признаков...


In [6]:
X = train_data_with_features.drop('rate', axis=1)
y = train_data['rate']

X_test = test_data["text"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
model = CatBoostClassifier(
    iterations=1000,              # количество деревьев
    learning_rate=0.02,          # скорость обучения
    depth=7,                     # глубина деревьев
    l2_leaf_reg=3,               # L2 регуляризация
    border_count=128,            # количество границ для числовых признаков
    random_seed=42,
    early_stopping_rounds=50,    # ранняя остановка
    eval_metric='MultiClass',    # используем MultiClass для валидации
    loss_function='MultiClass',  # функция потерь для многоклассовой классификации
    verbose=100,
    task_type='CPU',
    custom_metric=['F1', 'Accuracy']         # F1 считаем дополнительно, но не используем для ранней остановки
)

train_pool = Pool(
    data=X_train.values,        # текст как массив строк
    label=y_train.values,
    text_features=[0]           # указываем, что признак с индексом 0 — текстовый
)

val_pool = Pool(
    data=X_val.values,
    label=y_val.values,
    text_features=[0]
)

model.fit(
    train_pool,
    eval_set=val_pool,
    verbose=100,
    plot=False
)

0:	learn: 1.5804353	test: 1.5798820	best: 1.5798820 (0)	total: 4.03s	remaining: 1h 7m 4s
100:	learn: 0.9687324	test: 0.9567094	best: 0.9567094 (100)	total: 7m 42s	remaining: 1h 8m 41s
200:	learn: 0.9183320	test: 0.9101428	best: 0.9101428 (200)	total: 15m 7s	remaining: 1h 5s
300:	learn: 0.9040210	test: 0.8996607	best: 0.8996607 (300)	total: 22m 31s	remaining: 52m 18s
400:	learn: 0.8947599	test: 0.8939806	best: 0.8939806 (400)	total: 29m 57s	remaining: 44m 44s
500:	learn: 0.8869253	test: 0.8895845	best: 0.8895845 (500)	total: 37m 14s	remaining: 37m 5s
600:	learn: 0.8780388	test: 0.8849986	best: 0.8849986 (600)	total: 44m 40s	remaining: 29m 39s
700:	learn: 0.8694967	test: 0.8812330	best: 0.8812330 (700)	total: 52m 25s	remaining: 22m 21s
800:	learn: 0.8621485	test: 0.8787801	best: 0.8787801 (800)	total: 59m 52s	remaining: 14m 52s
900:	learn: 0.8557997	test: 0.8769405	best: 0.8769405 (900)	total: 1h 7m 12s	remaining: 7m 23s
999:	learn: 0.8498537	test: 0.8753747	best: 0.8753747 (999)	total: 

CatBoostClassifier(border_count=128, custom_metric=['F1', 'Accuracy'], depth=7, early_stopping_rounds=50, eval_metric='MultiClass', iterations=1000, l2_leaf_reg=3, learning_rate=0.02, loss_function='MultiClass', random_seed=42, task_type='CPU', verbose=100)

# Predict

In [8]:
y_val_pred = model.predict(val_pool)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')

print(f"F1-score на валидации: {val_f1:.4f}")
print("\nОтчет по классификации на валидации:")
print(classification_report(y_val, y_val_pred))

print("\nТоп-20 самых важных слов:")
feature_importance = model.get_feature_importance()
feature_names = model.feature_names_
important_features = sorted(zip(feature_names, feature_importance),
                           key=lambda x: x[1], reverse=True)[:20]
for feat, imp in important_features:
    if feat != 'text':
        print(f"  {feat}: {imp:.4f}")

F1-score на валидации: 0.6135

Отчет по классификации на валидации:
              precision    recall  f1-score   support

           1       0.50      0.63      0.56       828
           2       0.00      0.00      0.00       482
           3       0.40      0.38      0.39      1225
           4       0.45      0.33      0.38      1984
           5       0.76      0.89      0.82      5214

    accuracy                           0.64      9733
   macro avg       0.42      0.45      0.43      9733
weighted avg       0.59      0.64      0.61      9733


Топ-20 самых важных слов:
  0: 74.8103
  119: 3.2208
  132: 2.7914
  266: 1.1049
  284: 1.1009
  157: 1.0737
  120: 0.6741
  3: 0.6070
  100: 0.6048
  161: 0.5554
  155: 0.5543
  127: 0.4779
  74: 0.4276
  121: 0.4026
  263: 0.3704
  31: 0.3539
  269: 0.3339
  126: 0.3160
  164: 0.3106
  30: 0.2938


In [9]:
# Preparing data in Pool format
dataset_test = Pool(
    data=test_data_with_features,
    text_features=[0]
)
predict_classes = model.predict(dataset_test)
predictions = predict_classes

# Create submission

In [10]:
sample_submission = pd.read_csv("sample_submission.csv")
sample_submission["rate"] = predictions
sample_submission.head()

,index,rate
0,0,5
1,1,5
2,2,5
3,3,4
4,4,5


In [11]:
sample_submission.to_csv("submission_catboost.csv", index=False)